# Week 13. GenAI APIs

Generative AI tools are proving to be quite powerful and beyond their chat capabilities, we can utilize the models for analysis tasks through their **API**. 

At AAP, our IT team has built a wonderful access point that connects to different GenAI APIs, but keeps the data in house. 

First, access the Portal endpoint here: [https://share.aap.cornell.edu/apps/oaiportal/portal.pl](https://share.aap.cornell.edu/apps/oaiportal/portal.pl)

Next, activate your API key. Immediately copy the key and add it to the `openai_key.py` file. 

IT has given each of us \\$10 in credits to use. Every time you use the API, it costs money in the form of tokens. You can see the costs for each model type [here](https://share.aap.cornell.edu/apps/oaiportal/portal.pl?action=docs). 

Costs are determined as dollars per token. A token is a representation of words, and roughly 1 token = 0.75 words (or 100 tokens = 75 words). 
- So if the input cost is \\$0.0025/1K tokens, then it costs about \\$1 for 400,000 tokens = roughly 300,000 words. This is a lot of words!
- Likewise, if the output cost is \\$0.01/1K tokens, then it costs about \\$1 for 100,000 tokens = roughly 75,000 words. 

What is an input? An input includes the prompt and any text data you pass forward. For example, if we build a prompt for our listings, we would have a prompt asking the model to label them single family, multi family, or ADU and pass all of the listings themselves. 

One catch with inputs is that the models can only handle so many tokens (or words) at one time. Right now, this limit is usually around 100,000 words. So we usually need to break up the input text (like the listings) into chunks and pass the prompt along with the chunk each time. This increases the cost slightly.

What is an output? This is whatever you ask of the model! For our example, the output would be a list of labels. 

## 1. Basic API Commands

In [ ]:
!pip install openai tiktoken

In [ ]:
#tiktoken is the encoding used by many models and can tell us how many tokens our text is
import tiktoken

# First we define what model we are going to use later
enc = tiktoken.encoding_for_model("gpt-4o")

# Then we encode or tokenize the text
text = df_train['body'].iloc[0]
tokens = enc.encode(text)

In [ ]:
# How many tokens? 
print(len(tokens))

In [ ]:
from openai import OpenAI
from openai_key import OPENAI_KEY

# We define the client up front
# Because we are using AAP's endpoint, we provide the base url, you do not need this directly from OpenAI
client = OpenAI(
    base_url="https://share.aap.cornell.edu/apps/oaigw/gw.pl/v1",
    api_key= OPENAI_KEY)

In [ ]:
# Pass it a simple prompt
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain quantum computing in simple terms"}])

print(response.choices[0].message.content)

## 2. Working with Data

Let's load in our listings data and build a prompt to identify what type of housing each one is.

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


data = pd.read_csv('labeled_data.csv')
data.head()

In [ ]:
# Remember we need to fix our labels

# First remove spaces
data['LABEL_clean'] = data['LABEL'].apply(lambda x: str(x).replace(" ",""))
# Change A to ADU ? 
data['LABEL_clean'] = data['LABEL_clean'].apply(lambda x: "ADU" if x == 'A' else x)

# Remove S ADU because we it doesn't fit nicely into one of our categories
data['LABEL_clean'] = data['LABEL_clean'].apply(lambda x: np.nan if x == 'SADU' or x == '' else x)

# drop these NaNs
data = data.dropna(subset='LABEL_clean')

data['LABEL_clean'].value_counts()

With GenAI models, we do not need to do heavy preprocessing. These models are built on massive collections on text that understand upper/lower casing, punctuation, and symbols. In fact, these nuances can help the model understand context and subtext (to an extent).

In [ ]:
# Get each training and testing set 
df_train, df_test = train_test_split(data, train_size=0.75, stratify=data["LABEL_clean"])

df_train.head()

Next, we can build out a prompt. This is like what you might put directly into a chat model. 

In [ ]:
prompt_preamble = f"""You are a researcher classifying rental housing listings according to
    the type of housing structure. The options are single-family house, multi-family house
    including duplexes and apartment buildings, additional dwelling units (ADUs) including
    garden homes or in-law suites, or unknown for any ambiguous listings. Listings may be for 
    renting the whole unit or for becoming a roommate in an existing renter situation. Regardless 
    of this intent, you are trying to identify the structure of the home itself.

    Respond to the listing with one of the following options:
    - M : for multi-family homes (apartments, duplexes, complexes)
    - S : for single-family homes (no shared walls)
    - ADU : for additional dweeling units (ADUs) (garden home, in-law suite)
    - U : for unknown or ambiguous listings in which the structure type cannot be found

    The listing text follows here: 
    
    """

In [ ]:
# We can add one of our listings on to the prompt

prompt = prompt_preamble + df_test['body'].iloc[1]

print(prompt)

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    input=prompt)

In [ ]:
response.output_text

In [ ]:
# Now we can run it for all of our listings in df_test

# Let's modify our prompt_preamble:

prompt = f"""You are a researcher classifying rental housing listings according to
    the type of housing structure. The options are single-family house, multi-family house
    including duplexes and apartment buildings, additional dwelling units (ADUs) including
    garden homes or in-law suites, or unknown for any ambiguous listings. Listings may be for 
    renting the whole unit or for becoming a roommate in an existing renter situation. Regardless 
    of this intent, you are trying to identify the structure of the home itself.

    Respond to the listing with one of the following options:
    M : for multi-family homes (apartments, duplexes, complexes)
    S : for single-family homes (no shared walls)
    ADU : for additional dweeling units (ADUs) (garden home, in-law suite)
    U : for unknown or ambiguous listings in which the structure type cannot be found

    Format your response as a python list where each element is one of the above responses 
    for each listing. It should look something like ['U','M','S'] only. No code block.
    The length of the list must exactly match the number of listings. 
    Do not include any explanations. 

    

    The listings are included below and the start of each listing is designated by:
    Listing #:

    The text of each listing follows here: 
    
    """

In [ ]:
# We can add the listings in succession for the first 100 rows

for idx, row in df_test[0:100].iterrows():
    prompt += f"""
    Listing {idx+1}:
    {row['body']}

    """

In [ ]:
print(len(prompt)) # Number of characters

# How many tokens?
enc = tiktoken.encoding_for_model("gpt-4o")

# Then we encode or tokenize the text
tokens = enc.encode(prompt)

len(tokens)

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    input=prompt)

In [ ]:
response.output_text

In [ ]:
labels = eval(response.output_text)
print(len(labels))

In [ ]:
# We need to make our prompt clearer:

prompt = f"""You are a researcher classifying rental housing listings according to
    the type of housing structure. The options are single-family house, multi-family house
    including duplexes and apartment buildings, additional dwelling units (ADUs) including
    garden homes or in-law suites, or unknown for any ambiguous listings. Listings may be for 
    renting the whole unit or for becoming a roommate in an existing renter situation. Regardless 
    of this intent, you are trying to identify the structure of the home itself.

    Respond to the listing with one of the following options:
    M : for multi-family homes (apartments, duplexes, complexes)
    S : for single-family homes (no shared walls)
    ADU : for additional dweeling units (ADUs) (garden home, in-law suite)
    U : for unknown or ambiguous listings in which the structure type cannot be found

    - Return EXACTLY N labels
    - The output MUST be a Python list
    - The list length MUST equal N
    - Do NOT skip or combine items
    - Do NOT include any explanation

    N = 100

    Format your response as a python dictionary where each element is one of the above responses 
    for each listing. It should look something like {{'Listing 1: 'U','Listing 2: 'M', 'Listing 3:'S'}} only. 
    No code block. The length of the list must exactly match the number of listings. 
    Do not include any explanations. 

    The listings are included below and the start of each listing is designated by:
    ###########################################################
    Listing #:

    The text of each listing follows here: 
    
    """
# Adding in the listings
for idx, row in df_test[0:100].iterrows():
    prompt += f"""
    ###########################################################
    Listing {idx+1}:
    {row['body']}

    """

In [ ]:
# Run the model
response = client.responses.create(
    model="gpt-4o",
    input=prompt)

In [ ]:
# Get back the results
labels = eval(response.output_text)
print(len(labels))

In [ ]:
# Compute the accuracy statistics
true_values = df_test["LABEL_clean"].iloc[0:100]
predicted_values = list(labels.values())
print(classification_report(true_values, predicted_values))

## 3. Prompt Engineering

Prompt engineering is the process of adjusting the written prompt to achieve desired outcomes. As a modeling strategry, it is nuanced, personal, and can heavily impact results (for better or worse). 

In [ ]:
# What if we add a few of our training examples to the prompt? 

prompt = f"""You are a researcher classifying rental housing listings according to
    the type of housing structure. The options are single-family house, multi-family house
    including duplexes and apartment buildings, additional dwelling units (ADUs) including
    garden homes or in-law suites, or unknown for any ambiguous listings. Listings may be for 
    renting the whole unit or for becoming a roommate in an existing renter situation. Regardless 
    of this intent, you are trying to identify the structure of the home itself.

    Respond to the listing with one of the following options:
    M : for multi-family homes (apartments, duplexes, complexes)
    S : for single-family homes (no shared walls)
    ADU : for additional dweeling units (ADUs) (garden home, in-law suite)
    U : for unknown or ambiguous listings in which the structure type cannot be found

    Here are three examples of multi-family (M) listings:
    {df_train[df_train['LABEL_clean'] == 'M']['body'].iloc[0]}
    {df_train[df_train['LABEL_clean'] == 'M']['body'].iloc[1]}
    {df_train[df_train['LABEL_clean'] == 'M']['body'].iloc[2]}

    Here are three examples of single-family (S) listings:
    {df_train[df_train['LABEL_clean'] == 'S']['body'].iloc[0]}
    {df_train[df_train['LABEL_clean'] == 'S']['body'].iloc[1]}
    {df_train[df_train['LABEL_clean'] == 'S']['body'].iloc[2]}
    
    Here are three examples of ADU (ADU) listings:
    {df_train[df_train['LABEL_clean'] == 'ADU']['body'].iloc[0]}
    {df_train[df_train['LABEL_clean'] == 'ADU']['body'].iloc[1]}
    {df_train[df_train['LABEL_clean'] == 'ADU']['body'].iloc[2]}

    Here are three examples of unknown (U) listings: 
    {df_train[df_train['LABEL_clean'] == 'U']['body'].iloc[0]}
    {df_train[df_train['LABEL_clean'] == 'U']['body'].iloc[1]}
    {df_train[df_train['LABEL_clean'] == 'U']['body'].iloc[2]}

    - Return EXACTLY N labels
    - The output MUST be a Python list
    - The list length MUST equal N
    - Do NOT skip or combine items
    - Do NOT include any explanation

    N = 100

    Format your response as a python dictionary where each element is one of the above responses 
    for each listing. It should look something like {{'Listing 1: 'U','Listing 2: 'M', 'Listing 3:'S'}} only. 
    No code block. The length of the list must exactly match the number of listings. 
    Do not include any explanations. 

    The listings are included below and the start of each listing is designated by:
    ###########################################################
    Listing #:

    The text of each listing follows here: 
    
    """
# Adding in the listings
for idx, row in df_test[0:100].iterrows():
    prompt += f"""
    ###########################################################
    Listing {idx+1}:
    {row['body']}

    """

In [ ]:
# How many tokens?
enc = tiktoken.encoding_for_model("gpt-4o")

# Then we encode or tokenize the text
tokens = enc.encode(prompt)

len(tokens)

In [ ]:
# Run the model
response = client.responses.create(
    model="gpt-4o",
    input=prompt)

In [ ]:
# Get back the results
labels = eval(response.output_text)
print(len(labels))

In [ ]:
# Compute the accuracy statistics
true_values = df_test["LABEL_clean"].iloc[0:100]
predicted_values = list(labels.values())
print(classification_report(true_values, predicted_values))